In [ ]:
#匯入套件與設定中繼資料
from pathlib import Path  # 路徑處理：用物件方式組合檔案路徑，比字串拼接更安全
from datetime import datetime, timezone  # 時間處理：記錄分析時間，timezone 確保時區正確
import hashlib  # 雜湊計算：對 CSV 做 SHA256，確認資料未被竄改、版本可追溯
import json  # JSON 處理：讀寫設定檔、輸出分析結果
import platform  # 系統資訊：記錄 OS / Python 版本，確保分析可重現
import sys  # 系統模組：查看 Python 版本、路徑
import warnings  # 警告控制：過濾或顯示套件的警告訊息

import numpy as np  # 數值運算：陣列、隨機數、統計計算核心
import pandas as pd  # 資料處理：讀 CSV、整理 DataFrame
import matplotlib.pyplot as plt  # 繪圖：管制圖、直方圖、常態檢定圖
import scipy  # 科學計算基礎庫
from scipy.stats import norm, shapiro  # norm：常態分佈計算；shapiro：Shapiro-Wilk 常態性檢定
import statsmodels  # 進階統計建模庫
import statsmodels.formula.api as smf  # 公式式建模介面：做 ANOVA / 迴歸，像 R 的 lm(y~x)
from statsmodels.stats.anova import anova_lm  # 變異數分析表：MSA 中拆解人員/零件變異用
from IPython.display import display, HTML, Image  # Jupyter 顯示：美化表格(HTML)、顯示圖片

# ---------------------------------------------------------
# 檔案路徑：預設 CSV 與 Notebook 放在同一資料夾
# ---------------------------------------------------------
BASE_DIR = Path(".")  # 基準資料夾："." 代表 Notebook 目前所在目錄
MSA_FILE = BASE_DIR / "msa_synthetic_data.csv"  # MSA 量測系統分析用的模擬資料檔
PROCESS_FILE = BASE_DIR / "process_synthetic_data.csv"  # SPC 製程管制用的模擬資料檔

# ---------------------------------------------------------
# 製程中繼資料：請依實際案例修改
# ---------------------------------------------------------
META = {  # 字典：集中存放本案例的規格與背景資訊，後續計算 Cp/Cpk 會直接引用
    "characteristic": "軸徑",  # 管制特性名稱
    "unit": "mm",  # 單位：毫米
    "LSL": 19.900,  # 規格下限 Lower Spec Limit
    "USL": 20.100,  # 規格上限 Upper Spec Limit，公差 = 0.2mm
    "target": 20.000,  # 目標值 / 中心值
    "measurement_system_id": "GAGE-DEMO-01",  # 量具編號：追溯是哪一把量具
    "data_version": "teaching-demo-1.0",  # 資料版本：教學示範版
    "data_source": "Python亂數產生的模擬資料",  # 資料來源聲明：非真實生產數據
}

# ---------------------------------------------------------
# 教學用分析設定；不是通用驗收標準
# ---------------------------------------------------------
MSA_MAX_PCT_TOLERANCE = 30.0  # MSA 判定門檻：%Tolerance ≤30% 才算可接受 (AIAG 參考，教學用)
MSA_MIN_NDC = 5  # NDC 鑑別力門檻：≥5 才表示量具能區分至少 5 個等級
MIN_SUBGROUPS = 20  # SPC 最小子組數：X-bar/R 圖至少要 20 組才穩定
BOOTSTRAP_REPS = 1000  # Bootstrap 重抽樣次數：用於估計信賴區間
RANDOM_SEED = 2026  # 隨機種子：固定亂數，確保每次執行結果相同、可重現

# X-bar / R 常數：本範例要求子組大小 n=4
A2 = 0.729  # X-bar 圖管制界限係數 (n=4 時查表值)
D3 = 0.000  # R 圖下管制界限係數 (n=4 時為 0，表示下限為 0)
D4 = 2.282  # R 圖上管制界限係數 (n=4 時查表值)
d2 = 2.059  # 估計標準差用常數：sigma = R-bar / d2 (n=4 時)

rng = np.random.default_rng(RANDOM_SEED)  # 建立可重現的隨機數產生器，後續抽樣都用它

errors = []  # 收集錯誤訊息：流程中斷時記錄原因
warnings_list = []  # 收集警告訊息：例如樣本不足、常態性不通過
decisions = []  # 收集判定結果：例如 MSA 通過/不通過，供最後總結報告用

In [ ]:
def sha256_file(path):
    """計算輸入檔案的 SHA-256，供追溯使用。"""
    # 功能：計算檔案雜湊值，確認資料版本、追溯是否被修改
    digest = hashlib.sha256()  # 建立 SHA-256 雜湊物件
    with open(path, "rb") as f:  # 二進位唯讀開啟，大檔也適用
        for block in iter(lambda: f.read(65536), b""):  # 每次讀 64KB 分塊讀，避免佔滿記憶體
            digest.update(block)  # 累計更新雜湊值
    return digest.hexdigest()  # 回傳十六進位字串


def check_unit(df, dataset_name):
    """
    前一版模擬 CSV 沒有 unit 欄。
    此處依 META 的單位宣告，但提醒使用者確認原始資料單位。
    """
    # 功能：檢查單位一致性，避免 mm 誤當 cm/inch 導致 Cp/Cpk 算錯
    if "unit" not in df.columns:  # 舊版 CSV 無 unit 欄的相容處理
        warnings_list.append(  # 只記警告，沿用 META 宣告單位
            f"{dataset_name} 缺少 unit 欄；"
            f"本次依中繼資料宣告單位為 {META['unit']}，請確認原始資料。"
        )
        return  # 提前結束

    units = df["unit"].dropna().astype(str).str.strip().unique()  # 去空值→轉字串→去空白→取唯一值，如 ['mm']

    if len(units) == 0:  # 全空
        errors.append(f"{dataset_name} 的 unit 欄沒有有效值。")
    elif len(units) > 1:  # 混用多種單位
        errors.append(f"{dataset_name} 出現多種單位：{units.tolist()}")
    elif units[0] != META["unit"]:  # 與 META 設定不一致
        errors.append(
            f"{dataset_name} 單位為 {units[0]}，"
            f"但中繼資料設定為 {META['unit']}。"
        )


def require_file(path):
    # 功能：檢查必要 CSV 是否存在，不存在就拋出清楚指引
    if not path.exists():
        raise FileNotFoundError(
            f"找不到 {path.resolve()}。"  # 顯示絕對路徑方便除錯
            "請先執行前一個範例產生 CSV，並確認檔案與 Notebook 位於同一資料夾。"
        )


require_file(MSA_FILE)  # 檢查 MSA 資料檔是否存在
require_file(PROCESS_FILE)  # 檢查製程資料檔是否存在

msa = pd.read_csv(MSA_FILE)  # 讀入 MSA 資料：part(零件)/operator(人員)/repeat(重複)/measurement(量測值)
process = pd.read_csv(PROCESS_FILE)  # 讀入製程資料：subgroup/measurement 等 SPC 用欄位

print("MSA 資料預覽")
display(msa.head())  # 顯示前 5 列，快速確認格式

print("製程資料預覽")
display(process.head())  # 顯示製程資料前 5 列

print("MSA 欄位：", msa.columns.tolist())  # 列出 MSA 欄名，檢查缺欄
print("製程資料欄位：", process.columns.tolist())  # 列出製程欄名

check_unit(msa, "MSA 資料")  # 檢查 MSA 單位
check_unit(process, "製程資料")  # 檢查製程單位

# MSA 必要欄位
msa_required = {"part", "operator", "repeat", "measurement"}  # MSA 最少需要的 4 欄
missing = msa_required - set(msa.columns)  # 集合差集找出缺少的欄位

if missing:  # 有缺欄
    errors.append(f"MSA 缺少欄位：{sorted(missing)}")  # 後續 ANOVA 不能做
else:
    msa["measurement"] = pd.to_numeric(msa["measurement"], errors="coerce")  # 轉數字，不能轉變 NaN 以揪髒資料

    if msa[list(msa_required)].isna().any().any():  # 檢查必要欄有無缺值/非數值
        errors.append("MSA 必要欄位有缺值或非數值資料。")

    duplicated_msa = msa.duplicated(  # 檢查重複：同零件+同人+同次不應出現兩筆
        subset=["part", "operator", "repeat"],  # 只比這三欄組合
        keep=False  # 所有重複列都標 True
    )
    if duplicated_msa.any():  # 有重複
        errors.append(
            f"MSA 有 {duplicated_msa.sum()} 筆重複的 "  # sum() 算出重複筆數
            "part/operator/repeat 組合。"
        )

    # 檢查平衡交叉設計
    counts = msa.groupby(["part", "operator"], observed=True).size()  # 算每個零件×人員組合有幾筆重複量測

    if counts.nunique() != 1:  # 各組合筆數不一致
        errors.append("MSA 不是平衡交叉設計：不同組合的重複數不一致。")  # 非平衡會影響 ANOVA 估計

    if not counts.empty and counts.iloc[0] < 2:  # 每組合少於 2 次
        errors.append("每個零件 × 作業員組合至少需有 2 次量測。")  # 不足 2 次無法估計重複性


# 製程資料必要欄位
process_required = {"subgroup", "measurement"}  # SPC 最少需要：子組編號 + 量測值
missing = process_required - set(process.columns)  # 找出缺少的欄位

if missing:
    errors.append(f"製程資料缺少欄位：{sorted(missing)}")
else:
    process["measurement"] = pd.to_numeric(  # 量測值轉數字
        process["measurement"], errors="coerce"
    )

    if process[list(process_required)].isna().any().any():  # 檢查缺值
        errors.append("製程資料的 subgroup 或 measurement 有缺值。")

    if "timestamp" not in process.columns:  # 無時間欄
        warnings_list.append(
            "製程資料沒有 timestamp 欄；本次只能按子組順序檢查，"  # 只能看順序，無法追班別/時間趨勢
            "無法依實際時間、班別或批次追溯。"
        )

    for col in ["machine_id", "batch_id"]:  # 檢查可做分層分析的欄位
        if col not in process.columns:
            warnings_list.append(
                f"製程資料沒有 {col} 欄；無法比較不同機台或批次。"  # 少了就不能做機台間比較
            )


# 檢查規格設定
if META["LSL"] >= META["USL"]:  # 下限不可 ≥ 上限
    errors.append("規格設定錯誤：LSL 必須小於 USL。")  # 規格設反，Cp/Cpk 全錯

if not errors:  # errors 為空串列
    print("資料檢核：未發現阻止分析的錯誤。")  # 綠燈，可繼續 MSA/SPC
else:
    print("資料檢核：發現錯誤，後續分析應停止。")  # 紅燈，先修資料

print("\n警告：")  # 印出所有警告，不擋分析但要留意
for item in warnings_list:
    print(" -", item)

print("\n錯誤：")  # 印出所有錯誤，有錯就要停
for item in errors:
    print(" -", item)

In [ ]:
## MSA：交叉式 Gage R&R
#
#本資料採交叉設計：每一個零件都由每一位作業員重複量測。
#
#ANOVA 模型：
#
#Y = 零件效果 + 作業員效果 + 零件×作業員交互作用 + 重複性誤差
#
#本範例使用平衡交叉 ANOVA 的方法矩估計變異成分；若估計結果為負值，教學程式將該變異成分截為 0。
#
#注意：
#- 應同時看變異成分、圖形與量測作業情境。
#- 不要只用 ANOVA 的 p 值判定量測系統可不可用。
#- 本範例的 %Tolerance 與 ndc 門檻是可修改的課堂設定，不是通用驗收標準。


In [ ]:
#分析 Gage R&R
if errors:  # 前面資料檢核有錯
    raise ValueError(  # 直接中斷，不讓錯誤資料流入 MSA
        "資料檢核未通過，請先修正 Cell 4 列出的錯誤。"
    )


def analyze_crossed_grr(data):
    # 功能：交叉型 Gage R&R 分析，用雙因子 ANOVA 拆解變異來源
    # data：需含 part(零件)/operator(人員)/measurement(量測值)
    model = smf.ols(  # 建立 OLS 線性模型，公式同 R 的 lm()
        "measurement ~ C(part) * C(operator)",  # C()表類別因子，* = 主效應+交互作用：零件+人員+零件×人員
        data=data
    ).fit()  # 配適模型

    table = anova_lm(model, typ=2)  # 產生 Type II ANOVA 表，得到各項的 sum_sq(離均差平方和)/df(自由度)

    # statsmodels 版本可能提供或不提供 mean_sq；
    # 若未提供，手動以 sum_sq / df 計算。
    if "mean_sq" not in table.columns:  # 舊版沒有均方欄的相容處理
        table["mean_sq"] = table["sum_sq"] / table["df"]  # 均方 = 平方和 / 自由度，後續估變異用

    expected_terms = [  # ANOVA 表應有的 4 列
        "C(part)",  # 零件效應：反映真實產品差異
        "C(operator)",  # 人員效應：不同人量測的系統偏差
        "C(part):C(operator)",  # 交互作用：特定人量特定件的特殊偏差
        "Residual",  # 殘差：重複性 (同一人量同一件的隨機誤差)
    ]

    missing_terms = [term for term in expected_terms if term not in table.index]  # 檢查缺了哪一列
    if missing_terms:
        raise ValueError(  # 缺列通常是水準數不足 (如只有1人)，無法估計
            "ANOVA 表缺少預期項目："
            f"{missing_terms}\n實際列名稱：{table.index.tolist()}"
        )

    n_parts = data["part"].nunique()  # 零件數：有幾種零件
    n_operators = data["operator"].nunique()  # 人員數：有幾位作業員
    n_repeats = int(  # 每組合重複次數：假設平衡設計，取第一組筆數即可
        data.groupby(["part", "operator"], observed=True).size().iloc[0]
    )

    ms_part = table.loc["C(part)", "mean_sq"]  # 取出零件均方
    ms_operator = table.loc["C(operator)", "mean_sq"]  # 取出人員均方
    ms_interaction = table.loc["C(part):C(operator)", "mean_sq"]  # 取出交互作用均方
    ms_repeatability = table.loc["Residual", "mean_sq"]  # 取出殘差均方 = 重複性

    # 平衡交叉 ANOVA 的方法矩估計
    var_repeatability = max(ms_repeatability, 0.0)  # 重複性變異 = 殘差均方，max防負值
    var_interaction = max(  # 交互作用變異 = (MS交互-MS殘差)/重複數
        (ms_interaction - ms_repeatability) / n_repeats,
        0.0  # 理論上變異不為負，負值截斷為 0
    )
    var_operator = max(  # 人員變異 = (MS人員-MS交互)/(零件數×重複數)
        (ms_operator - ms_interaction) / (n_parts * n_repeats),
        0.0
    )
    var_part = max(  # 零件變異 = (MS零件-MS交互)/(人數×重複數)，代表真實製程變異
        (ms_part - ms_interaction) / (n_operators * n_repeats),
        0.0
    )

    var_reproducibility = var_operator + var_interaction  # 再現性 = 人員 + 交互作用
    var_grr = var_repeatability + var_reproducibility  # GRR 總量測變異 = 重複性 + 再現性
    var_total = var_grr + var_part  # 總變異 = 量測變異 + 零件變異

    sd_repeatability = np.sqrt(var_repeatability)  # 開根號變標準差，單位回到 mm
    sd_operator = np.sqrt(var_operator)  # 人員 SD
    sd_interaction = np.sqrt(var_interaction)  # 交互作用 SD
    sd_grr = np.sqrt(var_grr)  # GRR SD：量測系統總標準差
    sd_part = np.sqrt(var_part)  # 零件 SD
    sd_total = np.sqrt(var_total)  # 總 SD

    tolerance = META["USL"] - META["LSL"]  # 公差 = 上限-下限，本例 20.1-19.9=0.2mm

    pct_study = (  # %StudyVar：量測變異佔總變異比例，越小越好
        100 * sd_grr / sd_total if sd_total > 0 else np.nan  # 防除以零
    )
    pct_tolerance = (  # %Tolerance：6*SD_grr / 公差，AIAG 常用判定指標
        100 * 6 * sd_grr / tolerance if tolerance > 0 else np.nan
    )
    ndc = (  # NDC 可區分數：量具能分辨幾個等級，越大越好
        int(np.floor(1.41 * sd_part / sd_grr))  # 公式 1.41*SD_part/SD_grr 取整
        if sd_grr > 0 else 0
    )

    msa_pass = (  # 教學用通過閘門：兩條件都要滿足
        np.isfinite(pct_tolerance)  # %Tolerance 可計算非無限大
        and pct_tolerance <= MSA_MAX_PCT_TOLERANCE  # ≤30%
        and ndc >= MSA_MIN_NDC  # NDC ≥5
    )

    return {  # 打包所有結果回傳，方便後面列印與繪圖
        "anova": table,  # ANOVA 表
        "model": model,  # 迴歸模型物件
        "n_parts": n_parts,
        "n_operators": n_operators,
        "n_repeats": n_repeats,
        "sd_repeatability": sd_repeatability,
        "sd_operator": sd_operator,
        "sd_interaction": sd_interaction,
        "sd_grr": sd_grr,
        "sd_part": sd_part,
        "sd_total": sd_total,
        "pct_study": pct_study,
        "pct_tolerance": pct_tolerance,
        "ndc": ndc,
        "msa_pass": bool(msa_pass),
    }


msa_result = analyze_crossed_grr(msa)  # 對 MSA 資料執行分析，結果存字典

print(f"零件數：{msa_result['n_parts']}")  # 顯示設計規模
print(f"作業員數：{msa_result['n_operators']}")
print(f"每組重複數：{msa_result['n_repeats']}")
print(f"重複性 SD：{msa_result['sd_repeatability']:.6f} {META['unit']}")  # 取 6 位小數+單位
print(f"作業員變異 SD：{msa_result['sd_operator']:.6f} {META['unit']}")
print(f"交互作用 SD：{msa_result['sd_interaction']:.6f} {META['unit']}")
print(f"Gage R&R SD：{msa_result['sd_grr']:.6f} {META['unit']}")
print(f"%Study Variation：{msa_result['pct_study']:.2f}%")  # 佔總變異 %
print(f"%Tolerance：{msa_result['pct_tolerance']:.2f}%")  # 佔公差 %
print(f"ndc：{msa_result['ndc']}")  # 可區分數
print("教學用 MSA 閘門：", "通過" if msa_result["msa_pass"] else "未通過")  # 總判定

print("\nANOVA 表")
display(msa_result["anova"].round(6))  # 顯示 ANOVA 表，四捨五入到 6 位

In [ ]:
#MSA 互動圖與分析閘門
means = (  # 計算每個 零件×人員 組合的平均值，用來畫互動圖
    msa.groupby(["part", "operator"], observed=True)["measurement"]  # 依零件+人員分組，取量測值
    .mean()  # 算平均：消除重複量測的隨機誤差，看系統性偏差
    .unstack("operator")  # 把 operator 從縱向轉成橫向欄位，變成：列=零件，欄=人員，方便一條線代表一人
)

fig, ax = plt.subplots(figsize=(9, 5))  # 建立 9x5 吋圖框，ax 為繪圖區

for operator in means.columns:  # 逐位作業員畫一條折線
    ax.plot(
        means.index.astype(str),  # X軸：零件編號轉字串，避免被當連續數值
        means[operator],  # Y軸：該人員量各零件的平均值
        marker="o",  # 每個點用圓圈標示
        label=f"Operator {operator}"  # 圖例標籤：Operator A/B/C
    )

ax.set_title("MSA: Part × Operator Mean")  # 標題：零件×人員平均互動圖
ax.set_xlabel("Part")  # X軸標籤：零件
ax.set_ylabel(f"Mean measurement ({META['unit']})")  # Y軸標籤：平均量測值+單位 mm
ax.grid(alpha=0.25)  # 顯示淡格線，方便對齊讀值
ax.legend()  # 顯示圖例區分不同人員
plt.tight_layout()  # 自動調整邊距，避免標籤被切掉
plt.show()  # 顯示圖形
# 解讀：若各線大致平行→無明顯交互作用；若線交叉/分開→有人員偏差或交互作用，需改善量測手法

if msa_result["msa_pass"]:  # 前一格算出的 MSA 是否通過 (%Tolerance≤30% 且 NDC≥5)
    decisions.append("MSA 通過本 Notebook 設定的教學用閘門。")  # 記入決策清單，供最後總結報告用
    print("決策：MSA 通過教學用閘門，可繼續檢查製程穩定性。")  # 綠燈：量具可信，往下做 SPC
else:  # 未通過
    decisions.append("MSA 未通過教學用閘門；停止能力分析。")  # 記入決策清單
    print("決策：MSA 未通過教學用閘門，不進行製程能力分析。")  # 紅燈：量具不可信，先修量具，Cp/Cpk 算了也沒意義

In [ ]:
#X-bar / R 管制圖
def analyze_xbar_r(data):
    # 功能：計算 X-bar(平均數)/R(全距) 管制圖中心線與 3σ 界限，並找出界外子組
    group_sizes = data.groupby("subgroup", observed=True).size()  # 算每子組有幾筆，檢查分組

    if len(group_sizes) < MIN_SUBGROUPS:  # 子組數不足
        raise ValueError(  # 中斷避免界限不穩
            f"子組數 {len(group_sizes)} 少於設定值 {MIN_SUBGROUPS}。"  # 本例要求至少20組
        )

    if group_sizes.nunique() != 1:  # 各組筆數不一致
        raise ValueError("子組大小不一致，不能直接使用本範例的 X-bar/R 常數。")  # A2/D3/D4 依 n 查表

    subgroup_size = int(group_sizes.iloc[0])  # 取出每組 n

    if subgroup_size != 4:  # 本範例只備 n=4 常數
        raise ValueError(
            f"本 Notebook 範例使用 n=4 的常數，目前每組 n={subgroup_size}。"
        )

    summary = (  # 計算每子組平均與全距
        data.groupby("subgroup", observed=True)["measurement"]
        .agg(
            xbar="mean",  # 組平均：反映製程中心
            range=lambda x: x.max() - x.min()  # 組全距：最大值-最小值，反映組內變異
        )
        .reset_index()  # subgroup 變回欄位方便繪圖
        .sort_values("subgroup")  # 按子組順序排，管制圖時間才正確
    )

    xbarbar = summary["xbar"].mean()  # 總平均 X-double-bar：X-bar 圖中心線 CL
    rbar = summary["range"].mean()  # 平均全距 R-bar：R 圖中心線 CL

    xbar_ucl = xbarbar + A2 * rbar  # X-bar 上界 UCL，A2=0.729(n=4)
    xbar_lcl = xbarbar - A2 * rbar  # X-bar 下界 LCL
    r_ucl = D4 * rbar  # R 上界，D4=2.282
    r_lcl = D3 * rbar  # R 下界，D3=0.0 故下限為0

    xbar_ooc = (  # X-bar 界外遮罩：高於上界或低於下界
        (summary["xbar"] > xbar_ucl)
        | (summary["xbar"] < xbar_lcl)
    )
    r_ooc = (  # R 界外遮罩
        (summary["range"] > r_ucl)
        | (summary["range"] < r_lcl)
    )

    return {  # 打包回傳
        "summary": summary,  # 每組 xbar/range 明細
        "subgroup_size": subgroup_size,  # 每組 n
        "xbarbar": xbarbar,  # X-bar 中心線
        "rbar": rbar,  # R 中心線
        "xbar_ucl": xbar_ucl,
        "xbar_lcl": xbar_lcl,
        "r_ucl": r_ucl,
        "r_lcl": r_lcl,
        "xbar_ooc": summary.loc[xbar_ooc, "subgroup"].tolist(),  # X-bar 界外子組編號清單
        "r_ooc": summary.loc[r_ooc, "subgroup"].tolist(),  # R 界外子組編號清單
        "stable_3sigma": not (xbar_ooc.any() or r_ooc.any()),  # 兩圖皆無界外點才算穩定
    }


if msa_result["msa_pass"]:  # 只有 MSA 通過才做 SPC，避免用不可信量具判斷
    control_result = analyze_xbar_r(process)  # 對製程資料執行 X-bar/R 分析
    s = control_result["summary"]  # 取出明細表，簡寫為 s
    x = s["subgroup"]  # 取出子組編號為 X 軸

    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)  # 建上下兩張子圖共用 X 軸：上=X-bar，下=R

    axes[0].plot(x, s["xbar"], marker="o")  # 上圖：畫每子組平均折線
    axes[0].axhline(control_result["xbarbar"], color="green", label="CL")  # 畫中心線綠色實線
    axes[0].axhline(  # 畫上界紅色虛線
        control_result["xbar_ucl"], color="red", linestyle="--", label="UCL/LCL"
    )
    axes[0].axhline(  # 畫下界紅色虛線 (不重複加 label 避免圖例重複)
        control_result["xbar_lcl"], color="red", linestyle="--"
    )
    axes[0].set_title("X-bar Chart")  # 上圖標題
    axes[0].set_ylabel(f"Subgroup mean ({META['unit']})")  # Y軸：子組平均+單位
    axes[0].legend()  # 顯示 CL/UCL/LCL 圖例
    axes[0].grid(alpha=0.25)  # 淡格線方便讀值

    axes[1].plot(x, s["range"], marker="o")  # 下圖：畫每子組全距折線
    axes[1].axhline(control_result["rbar"], color="green", label="CL")  # R 中心線
    axes[1].axhline(  # R 上界
        control_result["r_ucl"], color="red", linestyle="--", label="UCL/LCL"
    )
    axes[1].axhline(control_result["r_lcl"], color="red", linestyle="--")  # R 下界
    axes[1].set_title("R Chart")  # 下圖標題
    axes[1].set_xlabel("Subgroup")  # X軸：子組順序
    axes[1].set_ylabel(f"Range ({META['unit']})")  # Y軸：全距+單位
    axes[1].legend()
    axes[1].grid(alpha=0.25)

    plt.tight_layout()  # 自動調邊距防標籤重疊
    plt.show()  # 顯示圖

    print("X-bar 圖界外子組：", control_result["xbar_ooc"])  # 列出平均失控的子組
    print("R 圖界外子組：", control_result["r_ooc"])  # 列出變異失控的子組
    print(  # 總結 3σ 穩定性
        "3-sigma 界限檢查：",
        "未發現界外點"  # stable_3sigma True
        if control_result["stable_3sigma"]
        else "發現界外點"  # 有界外點代表不穩定，需先找特殊原因
    )
else:  # MSA 沒過
    control_result = None  # 設為空，後續能力分析會跟著跳過
    print("因 MSA 閘門未通過，跳過管制圖與能力分析。")  # 提示：量具不可信不做 SPC

In [ ]:
#Code：能力指標與 bootstrap 信賴區間
def capability_metrics(values, sigma_within):
    values = np.asarray(values, dtype=float)

    lsl = META["LSL"]
    usl = META["USL"]
    mean = values.mean()
    sigma_overall = values.std(ddof=1)

    if sigma_within <= 0 or sigma_overall <= 0:
        raise ValueError("標準差估計必須大於 0。")

    cp = (usl - lsl) / (6 * sigma_within)
    cpu = (usl - mean) / (3 * sigma_within)
    cpl = (mean - lsl) / (3 * sigma_within)
    cpk = min(cpu, cpl)

    pp = (usl - lsl) / (6 * sigma_overall)
    ppu = (usl - mean) / (3 * sigma_overall)
    ppl = (mean - lsl) / (3 * sigma_overall)
    ppk = min(ppu, ppl)

    # 常態模型下的整體規格外機率估計
    p_below = norm.cdf(lsl, loc=mean, scale=sigma_overall)
    p_above = 1 - norm.cdf(usl, loc=mean, scale=sigma_overall)

    return {
        "mean": mean,
        "sigma_within": sigma_within,
        "sigma_overall": sigma_overall,
        "Cp": cp,
        "Cpu": cpu,
        "Cpl": cpl,
        "Cpk": cpk,
        "Pp": pp,
        "Ppu": ppu,
        "Ppl": ppl,
        "Ppk": ppk,
        "ppm_below_lsl": p_below * 1_000_000,
        "ppm_above_usl": p_above * 1_000_000,
        "ppm_total": (p_below + p_above) * 1_000_000,
    }


def bootstrap_capability(data, reps=1000):
    """
    以子組為單位重抽樣的教學用 percentile bootstrap。
    適用性仍需依資料結構和製程特性確認。
    """
    groups = [
        group["measurement"].to_numpy(dtype=float)
        for _, group in data.groupby("subgroup", observed=True)
    ]

    n_groups = len(groups)
    samples = {"Cp": [], "Cpk": [], "Pp": [], "Ppk": []}

    for _ in range(reps):
        selected = rng.integers(0, n_groups, size=n_groups)
        resampled_groups = [groups[i] for i in selected]
        resampled_values = np.concatenate(resampled_groups)

        resampled_rbar = np.mean([
            group.max() - group.min()
            for group in resampled_groups
        ])
        sigma_within = resampled_rbar / d2

        if sigma_within <= 0:
            continue

        result = capability_metrics(resampled_values, sigma_within)

        for name in samples:
            samples[name].append(result[name])

    intervals = {}
    for name, estimates in samples.items():
        if len(estimates) < reps * 0.5:
            intervals[name] = (np.nan, np.nan)
        else:
            intervals[name] = tuple(
                np.percentile(estimates, [2.5, 97.5])
            )

    return intervals


capability_result = None
capability_ci = None
normality_result = None

if msa_result["msa_pass"] and control_result is not None:
    if not control_result["stable_3sigma"]:
        decisions.append(
            "管制圖發現 3-sigma 界外點；能力分析停止。"
        )
        print("分析閘門：製程有界外點，本次不發布能力結論。")
    else:
        values = process["measurement"].to_numpy(dtype=float)
        sigma_within = control_result["rbar"] / d2

        capability_result = capability_metrics(values, sigma_within)
        capability_ci = bootstrap_capability(
            process,
            reps=BOOTSTRAP_REPS
        )

        # 常態性檢定只作為診斷資訊，不單獨決定分析方法
        if len(values) >= 3:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                normality_result = shapiro(values)

        decisions.append(
            "通過本範例 MSA 與 3-sigma 界限閘門，"
            "計算常態模型能力指標。"
        )

        print("樣本平均：", round(capability_result["mean"], 6))
        print("組內標準差估計：", round(capability_result["sigma_within"], 6))
        print("整體標準差：", round(capability_result["sigma_overall"], 6))
        print("Cp：", round(capability_result["Cp"], 3))
        print("Cpk：", round(capability_result["Cpk"], 3))
        print("Pp：", round(capability_result["Pp"], 3))
        print("Ppk：", round(capability_result["Ppk"], 3))
        print(
            "常態模型預估總不良率（ppm）：",
            round(capability_result["ppm_total"], 2)
        )

        if normality_result is not None:
            print("Shapiro-Wilk p 值：", round(normality_result.pvalue, 5))
            print(
                "提醒：不要只依 Shapiro-Wilk p 值決定分布，"
                "還要看直方圖、機率圖及製程知識。"
            )

        print("\nBootstrap 95% 區間（教學用）")
        display(pd.DataFrame(
            [
                {
                    "指標": name,
                    "下限": interval[0],
                    "上限": interval[1],
                }
                for name, interval in capability_ci.items()
            ]
        ).round(4))
else:
    print("因 MSA 閘門未通過，跳過能力分析。")


In [ ]:
#Code：能力指標與 bootstrap 信賴區間
def capability_metrics(values, sigma_within):
    # 功能：算 Cp/Cpk(組內) 與 Pp/Ppk(整體) 及常態模型預估 ppm
    # values：全部量測值陣列；sigma_within：由 R-bar/d2 估的組內標準差
    values = np.asarray(values, dtype=float)  # 轉成 float 陣列，確保可計算

    lsl = META["LSL"]  # 規格下限 19.9
    usl = META["USL"]  # 規格上限 20.1
    mean = values.mean()  # 樣本平均，製程中心位置
    sigma_overall = values.std(ddof=1)  # 整體標準差 (n-1 無偏估計)，含組間+組內變異

    if sigma_within <= 0 or sigma_overall <= 0:  # 標準差異常
        raise ValueError("標準差估計必須大於 0。")  # 防除以零，中斷分析

    cp = (usl - lsl) / (6 * sigma_within)  # Cp：潛在能力，只看寬度不看偏移，越大越好
    cpu = (usl - mean) / (3 * sigma_within)  # Cpu：上側能力，平均偏上會變小
    cpl = (mean - lsl) / (3 * sigma_within)  # Cpl：下側能力，平均偏下會變小
    cpk = min(cpu, cpl)  # Cpk：取較差側，兼顧中心與寬度

    pp = (usl - lsl) / (6 * sigma_overall)  # Pp：整體潛能，用整體標準差算
    ppu = (usl - mean) / (3 * sigma_overall)  # Ppu：整體上側
    ppl = (mean - lsl) / (3 * sigma_overall)  # Ppl：整體下側
    ppk = min(ppu, ppl)  # Ppk：整體實際能力，通常比 Cpk 小

    # 常態模型下的整體規格外機率估計
    p_below = norm.cdf(lsl, loc=mean, scale=sigma_overall)  # P(X<LSL)：低於下限機率，常態 CDF 算
    p_above = 1 - norm.cdf(usl, loc=mean, scale=sigma_overall)  # P(X>USL)：高於上限機率

    return {  # 打包回傳
        "mean": mean,
        "sigma_within": sigma_within,  # 組內 sigma
        "sigma_overall": sigma_overall,  # 整體 sigma
        "Cp": cp,
        "Cpu": cpu,
        "Cpl": cpl,
        "Cpk": cpk,
        "Pp": pp,
        "Ppu": ppu,
        "Ppl": ppl,
        "Ppk": ppk,
        "ppm_below_lsl": p_below * 1_000_000,  # 下超規 ppm = 機率×百萬
        "ppm_above_usl": p_above * 1_000_000,  # 上超規 ppm
        "ppm_total": (p_below + p_above) * 1_000_000,  # 總不良 ppm
    }


def bootstrap_capability(data, reps=1000):
    """
    以子組為單位重抽樣的教學用 percentile bootstrap。
    適用性仍需依資料結構和製程特性確認。
    """
    # 功能：用 Bootstrap 估 Cp/Cpk/Pp/Ppk 的 95% 信賴區間，呈現估計不確定性
    groups = [  # 把資料拆成子組 list，每元素是一組的量測陣列
        group["measurement"].to_numpy(dtype=float)
        for _, group in data.groupby("subgroup", observed=True)
    ]

    n_groups = len(groups)  # 子組數
    samples = {"Cp": [], "Cpk": [], "Pp": [], "Ppk": []}  # 存放每次重抽算出的指標

    for _ in range(reps):  # 重複 reps 次 (本例 1000)
        selected = rng.integers(0, n_groups, size=n_groups)  # 有放回隨機抽 n_groups 個組編號
        resampled_groups = [groups[i] for i in selected]  # 取出被抽到的組
        resampled_values = np.concatenate(resampled_groups)  # 併成一維大陣列，當作一次虛擬樣本

        resampled_rbar = np.mean([  # 算這次虛擬樣本的平均全距
            group.max() - group.min()  # 每組全距
            for group in resampled_groups
        ])
        sigma_within = resampled_rbar / d2  # 換算組內 sigma，d2=2.059(n=4)

        if sigma_within <= 0:  # 異常跳過本次
            continue

        result = capability_metrics(resampled_values, sigma_within)  # 算這次的 Cp/Cpk/Pp/Ppk

        for name in samples:  # 存入對應 list
            samples[name].append(result[name])

    intervals = {}  # 存各指標 95% 區間
    for name, estimates in samples.items():
        if len(estimates) < reps * 0.5:  # 成功次數太少 (<50%)
            intervals[name] = (np.nan, np.nan)  # 回傳空區間
        else:
            intervals[name] = tuple(  # percentile 法：取 2.5% 與 97.5% 分位數為上下限
                np.percentile(estimates, [2.5, 97.5])
            )

    return intervals


capability_result = None  # 初始化：能力指標結果，失敗時保持 None
capability_ci = None  # 初始化：bootstrap 區間
normality_result = None  # 初始化：常態檢定結果

if msa_result["msa_pass"] and control_result is not None:  # 雙閘門：MSA 過且管制圖有做
    if not control_result["stable_3sigma"]:  # 有界外點=不穩定
        decisions.append(  # 記決策
            "管制圖發現 3-sigma 界外點；能力分析停止。"  # 不穩定不能算 Cpk，需先消除特殊原因
        )
        print("分析閘門：製程有界外點，本次不發布能力結論。")
    else:  # 穩定才算能力
        values = process["measurement"].to_numpy(dtype=float)  # 取出全部量測值
        sigma_within = control_result["rbar"] / d2  # 組內 sigma = R-bar/d2

        capability_result = capability_metrics(values, sigma_within)  # 算能力指標
        capability_ci = bootstrap_capability(  # 算 bootstrap 95% 區間
            process,
            reps=BOOTSTRAP_REPS  # 1000 次
        )

        # 常態性檢定只作為診斷資訊，不單獨決定分析方法
        if len(values) >= 3:  # Shapiro 至少要 3 筆
            with warnings.catch_warnings():  # 暫時忽略警告
                warnings.simplefilter("ignore")
                normality_result = shapiro(values)  # Shapiro-Wilk 常態檢定

        decisions.append(
            "通過本範例 MSA 與 3-sigma 界限閘門，"  # 記決策：可發布常態模型結論
            "計算常態模型能力指標。"
        )

        print("樣本平均：", round(capability_result["mean"], 6))  # 印平均
        print("組內標準差估計：", round(capability_result["sigma_within"], 6))  # 印組內 sigma
        print("整體標準差：", round(capability_result["sigma_overall"], 6))  # 印整體 sigma
        print("Cp：", round(capability_result["Cp"], 3))  # 潛在能力
        print("Cpk：", round(capability_result["Cpk"], 3))  # 實際能力(組內)
        print("Pp：", round(capability_result["Pp"], 3))  # 整體潛能
        print("Ppk：", round(capability_result["Ppk"], 3))  # 整體實際能力
        print(
            "常態模型預估總不良率（ppm）：",  # 預估每百萬件不良數
            round(capability_result["ppm_total"], 2)
        )

        if normality_result is not None:  # 有做檢定才印
            print("Shapiro-Wilk p 值：", round(normality_result.pvalue, 5))  # p<0.05 表非常態
            print(
                "提醒：不要只依 Shapiro-Wilk p 值決定分布，"  # 教學提醒：大樣本易顯著，要搭配圖形
                "還要看直方圖、機率圖及製程知識。"
            )

        print("\nBootstrap 95% 區間（教學用）")  # 印區間表
        display(pd.DataFrame(  # 組成 DataFrame 顯示：指標/下限/上限
            [
                {
                    "指標": name,
                    "下限": interval[0],
                    "上限": interval[1],
                }
                for name, interval in capability_ci.items()
            ]
        ).round(4))  # 取 4 位小數
else:  # MSA 沒過
    print("因 MSA 閘門未通過，跳過能力分析。")  # 量具不可信不算能力

In [ ]:
#產生 HTML 報告與追溯紀錄
# 儲存圖表
msa_plot_file = BASE_DIR / "msa_interaction.png"  # MSA 互動圖存檔路徑
control_plot_file = BASE_DIR / "control_charts.png"  # X-bar/R 管制圖存檔路徑
capability_plot_file = BASE_DIR / "capability.png"  # 能力直方圖+常態曲線存檔路徑

# MSA 圖
means = (  # 算每 零件×人員 平均值，用來畫互動圖
    msa.groupby(["part", "operator"], observed=True)["measurement"]
    .mean()  # 平均消除重複誤差，看系統偏差
    .unstack("operator")  # operator 轉成欄：列=零件，欄=人員，一人一條線
)

fig, ax = plt.subplots(figsize=(8, 4.5))  # 建 8x4.5 吋圖框
for operator in means.columns:  # 逐人員畫線
    ax.plot(
        means.index.astype(str),  # X：零件編號轉字串
        means[operator],  # Y：該人員各零件平均
        marker="o",  # 圓點標示
        label=f"Operator {operator}"  # 圖例
    )
ax.set_title("MSA: Part × Operator Mean")  # 標題
ax.set_xlabel("Part")  # X軸：零件
ax.set_ylabel(f"Measurement ({META['unit']})")  # Y軸：量測值+單位
ax.legend()  # 顯示圖例
ax.grid(alpha=0.25)  # 淡格線
plt.tight_layout()  # 防標籤被切
fig.savefig(msa_plot_file, dpi=150)  # 存檔供 HTML 報告引用，dpi=150 清晰度適中
plt.close(fig)  # 關圖釋放記憶體，避免重複顯示

# 管制圖
if control_result is not None:  # 只有 MSA 通過、有做管制圖才畫
    s = control_result["summary"]  # 取出每組 xbar/range 明細，簡寫 s
    x = s["subgroup"]  # X軸：子組編號

    fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)  # 上下兩圖共用X軸：上=X-bar，下=R

    axes[0].plot(x, s["xbar"], marker="o")  # 上圖：子組平均折線
    axes[0].axhline(control_result["xbarbar"], color="green", label="CL")  # 中心線綠實線
    axes[0].axhline(  # 上界紅虛線
        control_result["xbar_ucl"], color="red", linestyle="--", label="UCL/LCL"
    )
    axes[0].axhline(  # 下界紅虛線，不重複加 label
        control_result["xbar_lcl"], color="red", linestyle="--"
    )
    axes[0].set_title("X-bar Chart")  # 上圖標題
    axes[0].set_ylabel(f"Mean ({META['unit']})")  # Y軸
    axes[0].legend()  # 圖例
    axes[0].grid(alpha=0.25)

    axes[1].plot(x, s["range"], marker="o")  # 下圖：子組全距折線
    axes[1].axhline(control_result["rbar"], color="green", label="CL")  # R 中心線
    axes[1].axhline(  # R 上界
        control_result["r_ucl"], color="red", linestyle="--", label="UCL/LCL"
    )
    axes[1].axhline(control_result["r_lcl"], color="red", linestyle="--")  # R 下界
    axes[1].set_title("R Chart")
    axes[1].set_xlabel("Subgroup")  # X軸：子組順序
    axes[1].set_ylabel(f"Range ({META['unit']})")
    axes[1].legend()
    axes[1].grid(alpha=0.25)

    plt.tight_layout()
    fig.savefig(control_plot_file, dpi=150)  # 存檔
    plt.close(fig)

# 能力圖
if capability_result is not None:  # 只有穩定、有算能力才畫
    values = process["measurement"].to_numpy(dtype=float)  # 取出全部量測值
    x = np.linspace(  # 產生 X 軸連續點用來畫常態曲線，範圍比資料/規格再寬 0.02
        min(values.min(), META["LSL"]) - 0.02,
        max(values.max(), META["USL"]) + 0.02,
        400  # 400 點畫平滑曲線
    )

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.hist(  # 畫直方圖看實際分布
        values,
        bins="auto",  # 自動決定組數
        density=True,  # Y軸為密度，才能跟常態 pdf 疊圖比較
        alpha=0.45,  # 半透明
        edgecolor="white",  # 白邊區分各柱
        label="Observed data"  # 實際資料
    )
    ax.plot(  # 疊加配適常態曲線
        x,
        norm.pdf(  # 常態 pdf
            x,
            capability_result["mean"],  # 用樣本平均為中心
            capability_result["sigma_overall"]  # 用整體 sigma 為寬度
        ),
        label="Fitted normal curve"
    )
    ax.axvline(META["LSL"], color="red", linestyle="--", label="LSL")  # 規格下限紅虛線
    ax.axvline(META["USL"], color="red", linestyle="--", label="USL")  # 規格上限紅虛線
    ax.axvline(META["target"], color="green", linestyle=":", label="Target")  # 目標值綠點線
    ax.set_title("Capability Plot")  # 標題
    ax.set_xlabel(f"Measurement ({META['unit']})")
    ax.set_ylabel("Density")
    ax.legend()
    ax.grid(alpha=0.2)
    plt.tight_layout()
    fig.savefig(capability_plot_file, dpi=150)  # 存檔
    plt.close(fig)


# 建立 HTML 表格
metadata_df = pd.DataFrame(  # 中繼資料轉表格：欄位/設定值兩欄，供報告第1節用
    [{"欄位": k, "設定值": v} for k, v in META.items()]
)

msa_summary_df = pd.DataFrame([  # MSA 重點結果彙整表
    ["零件數", msa_result["n_parts"]],
    ["作業員數", msa_result["n_operators"]],
    ["每組重複數", msa_result["n_repeats"]],
    ["Gage R&R SD", msa_result["sd_grr"]],  # 量測總標準差
    ["%Study Variation", msa_result["pct_study"]],  # 佔總變異比
    ["%Tolerance", msa_result["pct_tolerance"]],  # 佔公差比，判定用
    ["ndc", msa_result["ndc"]],  # 可區分數
    ["教學用 MSA 閘門", "通過" if msa_result["msa_pass"] else "未通過"],
], columns=["MSA 項目", "結果"])

warning_html = (  # 警告清單轉 HTML：有警告列 <ul>，無則顯示無警告
    "<ul>" + "".join(f"<li>{w}</li>" for w in warnings_list) + "</ul>"
    if warnings_list else "<p>無警告</p>"
)

decision_html = (  # 決策紀錄轉 HTML，供報告第6節
    "<ul>" + "".join(f"<li>{d}</li>" for d in decisions) + "</ul>"
    if decisions else "<p>無分析決策紀錄</p>"
)

if control_result is not None:  # 有做管制圖
    stability_text = (  # 穩定性文字結論
        "未發現 3-sigma 界外點"
        if control_result["stable_3sigma"]
        else "發現 3-sigma 界外點；能力結論不應直接發布"  # 不穩定警語
    )
    control_html = f"""  # 管制圖報告片段：文字+界外清單+圖片
    <p>管制圖結果：{stability_text}</p>
    <p>X-bar 圖界外子組：{control_result['xbar_ooc']}</p>
    <p>R 圖界外子組：{control_result['r_ooc']}</p>
    <img src="{control_plot_file.name}" width="850">
    """
else:  # 沒做 (MSA 未過)
    control_html = "<p>管制圖未執行。</p>"

if capability_result is not None:  # 有能力結果
    capability_df = pd.DataFrame([  # 能力指標彙整表
        ["平均值", capability_result["mean"]],
        ["組內標準差", capability_result["sigma_within"]],  # R-bar/d2
        ["整體標準差", capability_result["sigma_overall"]],  # 樣本標準差
        ["Cp", capability_result["Cp"]],
        ["Cpk", capability_result["Cpk"]],
        ["Pp", capability_result["Pp"]],
        ["Ppk", capability_result["Ppk"]],
        ["常態模型預估總不良率 (ppm)", capability_result["ppm_total"]],
    ], columns=["項目", "結果"])

    capability_html = f"""  # 能力報告片段：p值說明+指標表+能力圖
    <p>常態性檢查 p 值：
       {normality_result.pvalue:.5f}
       （若未執行則無資料）</p>
    <p>能力指標信賴區間採用子組重抽樣 bootstrap，
       屬本課程的教學示範方法。</p>
    {capability_df.to_html(index=False, float_format=lambda x: f"{x:.5f}")}  # 表格取5位小數
    <img src="{capability_plot_file.name}" width="850">
    """
else:  # 沒算 (不穩定或 MSA 未過)
    capability_html = "<p>能力分析未執行或未發布。</p>"

report_status = (  # 報告頂部狀態列文字
    "能力分析已完成（教學範例）"
    if capability_result is not None
    else "能力分析未完成或未發布"
)

html = f"""  # 組裝完整 HTML 報告字串，f-string 嵌入上面各片段
<!doctype html>
<html lang="zh-Hant">
<head>
<meta charset="utf-8">
<title>SPC / MSA 教學報告</title>
<style>
body {{font-family: Arial, sans-serif; margin: 32px; line-height: 1.6;}}  /* 內文樣式，{{}} 因 f-string 需跳脫 */
h1, h2 {{color: #075b70;}}  /* 標題深青色 */
table {{border-collapse: collapse; margin: 12px 0 24px;}}  /* 表格線合併 */
th, td {{border: 1px solid #bbb; padding: 7px 10px;}}  /* 儲存格框線+內距 */
th {{background: #e8f2f4;}}  /* 表頭淡藍底 */
.note {{background: #eef7f7; padding: 12px;}}  /* 限制說明區塊底色 */
.status {{background: #fff2d8; padding: 12px; font-weight: bold;}}  /* 狀態列黃底粗體 */
img {{max-width: 100%; height: auto;}}  /* 圖片自適應寬度 */
</style>
</head>
<body>
<h1>SPC / MSA 自動分析報告</h1>
<p class="status">報告狀態：{report_status}</p>
<p>產生時間：{datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds")}</p>  <!-- 本地時區 ISO 時間 -->

<h2>1. 中繼資料</h2>
{metadata_df.to_html(index=False)}

<h2>2. 資料檢核警告</h2>
{warning_html}

<h2>3. MSA 結果</h2>
{msa_summary_df.to_html(index=False, float_format=lambda x: f"{x:.5f}")}
<p><strong>方法：</strong>平衡交叉設計 ANOVA 方法矩估計。</p>
<img src="{msa_plot_file.name}" width="750">  <!-- 只取檔名，HTML 與圖同資料夾即可顯示 -->
<h3>ANOVA 表</h3>
{msa_result['anova'].to_html(float_format=lambda x: f"{x:.6g}")}

<h2>4. 製程穩定性</h2>
{control_html}

<h2>5. 製程能力分析</h2>
{capability_html}

<h2>6. 分析決策紀錄</h2>
{decision_html}

<h2>7. 限制與假設</h2>
<div class="note">
本 Notebook 是教學範例。管制圖目前只檢查 3-sigma 界外點，
沒有加入 Nelson／WECO 連續點規則。能力指標採常態模型，
Shapiro-Wilk 檢定 p 值不能單獨證明常態性。
MSA 門檻、樣本需求與能力判定標準需依實際用途及適用規範確認。
本範例的資料沒有時間戳記、機台和批次欄位，因此無法進行相應分層分析。
</div>
</body>
</html>
"""

report_file = BASE_DIR / "spc_msa_report.html"  # 報告輸出路徑
report_file.write_text(html, encoding="utf-8")  # 寫入 HTML 檔

run_log = {  # 建立追溯用 log 字典，記錄本次分析所有關鍵資訊
    "run_time_local": datetime.now().astimezone().isoformat(),  # 執行時間 (本地時區)
    "metadata": META,  # 規格中繼資料
    "input_sha256": {  # 輸入檔雜湊，證明用哪版資料算的
        str(MSA_FILE): sha256_file(MSA_FILE),
        str(PROCESS_FILE): sha256_file(PROCESS_FILE),
    },
    "warnings": warnings_list,  # 警告清單
    "decisions": decisions,  # 決策紀錄
    "settings": {  # 分析參數設定
        "msa_max_pct_tolerance": MSA_MAX_PCT_TOLERANCE,
        "msa_min_ndc": MSA_MIN_NDC,
        "min_subgroups": MIN_SUBGROUPS,
        "bootstrap_reps": BOOTSTRAP_REPS,
        "control_chart_rule": "僅檢查 3-sigma 界外點",  # 規則聲明
        "within_sigma_estimator": "R-bar / d2, n=4",  # 組內 sigma 估法
        "overall_sigma_estimator": "樣本標準差，ddof=1",  # 整體 sigma 估法
    },
    "software_versions": {  # 軟體版本，確保可重現
        "python": sys.version,
        "platform": platform.platform(),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "statsmodels": statsmodels.__version__,
    },
}

log_file = BASE_DIR / "analysis_log.json"  # log 輸出路徑
log_file.write_text(  # 寫入 JSON，ensure_ascii=False 保留中文，indent=2 美化排版
    json.dumps(run_log, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("已輸出：")  # 列印產出檔案清單方便確認
print(" -", report_file.resolve())  # 報告 HTML 絕對路徑
print(" -", log_file.resolve())  # 追溯 JSON 絕對路徑
print(" -", msa_plot_file.resolve())  # MSA 圖
if control_result is not None:  # 有管制圖才印
    print(" -", control_plot_file.resolve())
if capability_result is not None:  # 有能力圖才印
    print(" -", capability_plot_file.resolve())

In [ ]:
display(HTML(Path("spc_msa_report.html").read_text(encoding="utf-8")))
